# Visual Health — embeddings APTOS (validation externe)

Calcule les embeddings RETFound (`crop-mask`) des 3 662 images annotées
d'**APTOS 2019** (rétinopathie diabétique, Aravind Eye Hospital), une base
jamais utilisée pour le développement. L'évaluation se fait ensuite sur CPU,
en local (`scripts/external_validation.py`) : ce notebook ne fait que la
partie GPU (~10 min sur T4).

**Avant de lancer (Kaggle)** :
1. Une seule fois : ouvrir la page de la compétition
   [aptos2019-blindness-detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)
   et accepter son règlement (`Join Competition` / `I understand and accept`).
2. `Add Input` → onglet **Competitions** → `APTOS 2019 Blindness Detection`.
3. `Settings > Accelerator > GPU T4 x2`, `Settings > Internet > On`.
4. `Add-ons > Secrets` → `HF_TOKEN` coché (poids RETFound), sauf si un
   dataset avec `RETFound_mae_natureCFP.pth` est ajouté en input.
5. `Run All`, puis télécharger `visual_health_aptos_embeddings.zip` depuis le
   panneau `Output`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : activer un GPU T4"

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt

In [ ]:
import glob

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
os.makedirs(os.path.dirname(TARGET), exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/RETFound_mae_natureCFP.pth", recursive=True)) if ON_KAGGLE else []
if os.path.exists(TARGET):
    print("Poids RETFound déjà présents")
elif found:
    os.symlink(found[0], TARGET)
    print("Poids RETFound pris dans les inputs :", found[0])
else:
    # Token lu depuis les Secrets, jamais écrit dans le notebook
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
!ls -lh third_party/retfound/checkpoints

## Embeddings APTOS

Même extraction que `run_baseline.py` (normalisation ImageNet, RETFound
figé, `crop-mask`), cache avec empreinte du checkpoint et du prétraitement.
Écrit aussi `aptos_labels.csv` (clé, stade 0 à 4).

In [ ]:
import time

roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))})
assert roots, "APTOS introuvable : ajouter la compétition « APTOS 2019 Blindness Detection » en input"
APTOS_ROOT = roots[0]
CHECKPOINT = sorted(glob.glob("third_party/retfound/checkpoints/*.pth"))[0]
print(APTOS_ROOT, CHECKPOINT)

t0 = time.time()
!python -u scripts/extract_embeddings.py --dataset aptos --data-root "{APTOS_ROOT}" --checkpoint {CHECKPOINT} \
    --out-dir embeddings --batch-size 64 --num-workers 2 2>&1 | grep -v -i warning
out = "embeddings/aptos_crop-mask_224_RETFound_mae_natureCFP.npz"
if not os.path.exists(out) or os.path.getmtime(out) < t0 or not os.path.exists("embeddings/aptos_labels.csv"):
    raise RuntimeError("extract_embeddings.py a échoué (voir la sortie ci-dessus)")

## Garder le résultat

In [ ]:
import zipfile

archive = f"{WORK}/visual_health_aptos_embeddings.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in ("embeddings/aptos_crop-mask_224_RETFound_mae_natureCFP.npz", "embeddings/aptos_labels.csv"):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")
if not ON_KAGGLE:
    from google.colab import files
    files.download(archive)